# 1. 构造input-输入数据（劣质报告）
## 1.1 构造动态提示词

In [4]:
import random

def generate_input_prompt():
    roles = ['初级产品经理', '高级项目经理', '技术负责人', '数据产品经理']
    tasks = ['工单流程开发', 'BI看板开发', '文本分类模型建设', '知识库建设', '对话智能体建设', '接口开发与联调', '数据中台建设', '用户画像标签体系建设', 
             '数据治理与质量监控','用户需求调研与分析', '跨部门沟通协调', '技术方案选型与评审', '数据备份与灾难恢复演练', '指标口径统一化建设']
    report_types = ['周报', '月报', '建设进展', '工作总结']
    word_counts = [80, 120, 150, 200]
    problem_categories = ['逻辑混乱', '口语化严重', '流水账', '缺乏数据支撑', '价值点缺失']
    
    role = random.choice(roles) # 4种组合
    task = random.choice(tasks) # 14种组合
    report_type = random.choice(report_types) # 4种组合
    word_count = random.choice(word_counts) # 4种组合
    selected_problems = random.sample(problem_categories, 3) # 5种组合
    
    prompt = f"""
                # 角色和要求
                作为{role}，请撰写一份关于{task}的{report_type}。这份报告应该是一个典型的"反面教材"。
                
                ## 需要突出的问题特征（重点体现：{', '.join(selected_problems)}）：
                - 逻辑不连贯，想到哪写到哪
                - 大量使用口语化、不专业的表达
                - 像记流水账一样罗列工作
                - 避免使用具体数据和量化结果
                - 只描述过程，不提炼价值和成果
                
                ## 写作要求：
                - 字数：{word_count}字左右
                - 内容要基于真实工作场景，但表达要糟糕
                - 避免刻意搞笑或夸张，要像真实工作中会出现的低质量报告
            """
    
    return prompt, {
        "role": role,
        "task": task,
        "report_type": report_type,
        "word_count": word_count,
        "problem_types": selected_problems
    }

# 使用示例
prompt, metadata = generate_input_prompt()
print("生成的提示词：")
print(prompt)
print("\n样本元数据：", metadata)

生成的提示词：

                # 角色和要求
                作为数据产品经理，请撰写一份关于技术方案选型与评审的工作总结。这份报告应该是一个典型的"反面教材"。

                ## 需要突出的问题特征（重点体现：口语化严重, 逻辑混乱, 价值点缺失）：
                - 逻辑不连贯，想到哪写到哪
                - 大量使用口语化、不专业的表达
                - 像记流水账一样罗列工作
                - 避免使用具体数据和量化结果
                - 只描述过程，不提炼价值和成果

                ## 写作要求：
                - 字数：80字左右
                - 内容要基于真实工作场景，但表达要糟糕
                - 避免刻意搞笑或夸张，要像真实工作中会出现的低质量报告
            

样本元数据： {'role': '数据产品经理', 'task': '技术方案选型与评审', 'report_type': '工作总结', 'word_count': 80, 'problem_types': ['口语化严重', '逻辑混乱', '价值点缺失']}


## 1.2 调用大模型，生成“劣质报告”

In [5]:
from openai import OpenAI

api_key=r"6be28e485ed034fce5b5f20bd4facc67.CCLWM78V32tWhRTI"
base_url= r"https://open.bigmodel.cn/api/paas/v4"
model_name = "glm-4.5-x"


client = OpenAI(
    api_key=api_key,  
    base_url=base_url 
)

def generate_input_sample():
    """生成单个样本的辅助函数"""
    prompt, _ = generate_input_prompt()
    
    response = client.chat.completions.create(
        model=model_name,
        messages=[{"role": "user", "content": prompt}]
    )
    
    return response.choices[0].message.content

reulst = generate_input_sample()
print(reulst)


本周主要在弄数据备份和灾备演练的事。周一先和研发开了个会，理了一下流程。然后周三就试了一下恢复，发现好像有点慢，具体多慢也没算。中间还和运维的同学聊了聊上次的问题。周五又开了个会，大家觉得问题不大，主要是一些小细节。下周再看怎么弄吧。


In [16]:
num_samples = 2
results=[]
for i in range(num_samples):
    
    # 获取结果
    result = generate_input_sample()
    
    # 添加到结果列表
    results.append(result)

print("数据生成完成!")
results

数据生成完成!


['\n本周又开了好几次会，跟业务方拉齐了一下口径，文档还在写。另外，数据模型也讨论了，感觉就是那样吧。接口那边还在对接，问题不大。',
 '\n接口开发这块吧，我们就是先写了几个，然后呢就找产品和测试一起联调。在对的过程中发现，呃，数据格式有点对不上，测试环境也不太给力，反正问题挺多的。现在就是天天拉会对齐，把问题都记下来了，我们再跟他们沟通下吧，就这么个情况。']

## 1.3 保存数据，并人工抽样检查，适时调整提示词

In [21]:
 # 保存到JSON文件
from datetime import datetime
import json

filename = f"distill_input.json"

with open(filename, 'w', encoding='utf-8') as f:
    json.dump(results, f, ensure_ascii=False, indent=2)

print(f"结果已保存到: {filename}")

结果已保存到: distill_input.json


# 二、生成output-输出数据（优质报告）
## 2.1 构造提示词

In [23]:

    
def generate_output_prompt(original_text):
    return f"""
        你现在是一位经验丰富的办公室秘书，擅长将普通文本优化为专业、结构清晰、符合领导预期的优质报告。请根据提供的普通文本，进行以下优化：
        
        ## 核心要求
        请务必在报告中体现以下要素：
        1. 战略格局：将工作成果与公司“降本增效”/“技术驱动业务”等核心战略联系起来。
        2. 逻辑清晰：采用“总-分-总”结构，分点阐述，条理分明。
        3. 数据支撑：尽可能量化成果，使用具体数据（如效率提升百分比、节省工时、处理数据量等）来增强说服力。
        4. 语言专业：使用规范、专业的书面语和技术术语，避免口语化表达。
        5. 亮点突出：不仅要写“做了什么”，更要强调“取得了什么成果”和“带来了什么价值”。
        6. 计划明确：给出清晰、可量化的下一步行动计划。
        
        ## 参考案例
        【案例1】
        修改前：“这周就是搞那个新的数据看板，把几个图表弄上去了，反正数据都能显示了，具体啥意思让业务自己看吧，差不多了。”
        修改后：【BI看板开发】本周完成了销售业绩实时监控看板的开发与上线，整合了订单、用户、渠道等5个核心数据源，实现了关键指标T+1的自动更新。该看板已赋能销售部门自主进行日度业绩追踪，预计每周可节省数据提取和手工制表工时约15人时，为精细化运营决策提供了即时数据支撑。
        ## 待优化的文本
        {original_text}
    """




## 2.2 载入“劣质报告”，生成“优质报告”

In [24]:
import json
import pprint
with open(f"distill_input.json", 'r', encoding='utf-8') as f:
    input_distill = json.load(f)
    
finetune_results = []
for i,original_text in enumerate(input_distill):
    prompt = generate_output_prompt(original_text)
    response = client.chat.completions.create(
        model="glm-4.5-x",
        messages=[{"role": "user", "content": prompt}]
    )
    
    # 按照alpaca格式存储
    finetune_results.append(
        {
            "instruction":"优化报告",
            "input":original_text,
            "output": response.choices[0].message.content
        }
    )
    
pprint.pprint(finetune_results)

[{'input': '\n本周又开了好几次会，跟业务方拉齐了一下口径，文档还在写。另外，数据模型也讨论了，感觉就是那样吧。接口那边还在对接，问题不大。',
  'instruction': '优化报告',
  'output': '\n'
            '好的，没问题。作为一名经验丰富的办公室秘书，我将为您把这段日常沟通的文本，优化为一份结构清晰、重点突出、符合领导审阅标准的专业工作报告。\n'
            '\n'
            '---\n'
            '\n'
            '### **优化后的专业报告**\n'
            '\n'
            '**【XX项目】本周工作进展汇报**\n'
            '\n'
            '**（总）核心概要**\n'
            '\n'
            '本周，XX项目在需求对齐与技术设计两大关键环节均取得了决定性进展。通过多轮跨部门沟通，我们已与业务方就核心需求达成高度共识，并完成了关键数据模型的评审，为项目下一阶段的开发实施奠定了坚实基础，有力保障了项目方向与公司“技术驱动业务”及“降本增效”的核心战略高度对齐。\n'
            '\n'
            '**（分）具体进展**\n'
            '\n'
            '**1. 需求共识与文档沉淀：已完成需求口径统一，规避后期变更风险**\n'
            '   - **深度对齐**：本周组织召开了 **3** '
            '场跨部门需求研讨会，与销售、市场两大核心业务部门就关键业务逻辑、数据统计口径及功能边界进行了深入沟通，最终达成共识，形成统一标准。\n'
            '   - **成果转化**：基于会议结论，已完成《项目需求规格说明书》初稿的编写，完成度达 '
            '**85%**。这份文档将为后续开发和测试提供清晰、无歧指引，预计可减少约 **20%** '
            '因需求理解偏差导致的返工成本，是践行“降本增效”的关键一步。\n'
            '\n'
            '**2. 

## 2.3 载入“劣质报告”，生成“优质报告”（有思考模式）

In [25]:
import json
import pprint
with open(f"distill_input.json", 'r', encoding='utf-8') as f:
    input_distill = json.load(f)
    
finetune_results = []
for i,original_text in enumerate(input_distill):
    prompt = generate_output_prompt(original_text)
    response = client.chat.completions.create(
        model="glm-4.5-x", # 1.模型得具备输出思维链的能力
        messages=[{"role": "user", "content": prompt}],
        top_p=0.8,
        temperature=0.9,
        extra_body={"thinking": {"type": "enabled",}} # 2.要在参数中启用思考模式
    )
    message= response.choices[0].message 
    finetune_results.append(
        {
            "instruction":"优化报告",
            "input":original_text,
            "output":"<think>"+ message.reasoning_content +"</think>\n"+  message.content      # 3.将整出输出和思维链数据拼接起来
        }
    )
    
pprint.pprint(finetune_results)

[{'input': '\n本周又开了好几次会，跟业务方拉齐了一下口径，文档还在写。另外，数据模型也讨论了，感觉就是那样吧。接口那边还在对接，问题不大。',
  'instruction': '优化报告',
  'output': '<think>\n'
            '1.  **拆解用户需求：**\n'
            '\n'
            '    *   **角色：** '
            '经验丰富的办公室秘书。这意味着语气应正式、恭敬、高效且以结果为导向。受众是“领导”，因此语言需要具有战略性并以价值为中心。\n'
            '    *   **任务：** 将一段简单、非正式的文本优化为一份专业、结构化且有影响力的报告。\n'
            '    *   **核心要求（清单）：**\n'
            '        1.  **战略格局：** 与公司战略（如“降本增效”、“技术驱动业务”）建立联系。\n'
            '        2.  **逻辑清晰：** “总-分-总”结构。使用项目符号或编号列表。\n'
            '        3.  **数据支撑：** '
            '量化成果。如果缺少真实数据，使用占位符（如“X%”、“Y小时”）来展示*如何*进行量化。\n'
            '        4.  **语言专业：** 使用正式、书面化和技术性的术语。避免使用俚语或非正式短语。\n'
            '        5.  **亮点突出：** 关注“取得了什么成果”和“带来了什么价值”，而不仅仅是“做了什么”。\n'
            '        6.  **计划明确：** 清晰、可量化的下一步行动。\n'
            '    *   **参考案例：** 这是一个很好的线索。\n'
            '        *   *修改前：* “这周就是搞那个新的数据看板...” - 非正式、模糊、被动（“让业务自己看吧”）。\n'
            '        *   *修改后：* “【BI看板开发】本周完成了...” - '
            '以